# IDH from MRI: Colab launcher
Runs the code from GitHub (javencp/gliomas-idh-mri-prediction) on a Colab GPU.
No model code is written here: edit in VS Code, push, then re-run Setup step 2.

Data lives in Drive at `MyDrive/glioma/` (`processed.tar`, `manifest.csv`).
Results are copied back to Drive, then downloaded and committed from VS Code.

Before running: Runtime > Change runtime type > T4 GPU.

When finished: Runtime > Disconnect and delete runtime (stops compute-unit use).


## 1. Setup: run all three cells at the start of every new session
Safe to re-run. Takes a few minutes the first time because of the data conversion.

In [ ]:
# 1) Make your Drive visible to this temporary machine
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# 2) Get the latest code: clones on a fresh machine, pulls if the repo is already there
%cd /content
!git clone https://github.com/javencp/gliomas-idh-mri-prediction.git 2>/dev/null || git -C gliomas-idh-mri-prediction pull
%cd /content/gliomas-idh-mri-prediction

In [ ]:
# 3) Data: unpack the crops (skipped if already there), copy the manifest, make the fast .npy copy
!mkdir -p Data
!test -d Data/processed || tar -xf /content/drive/MyDrive/glioma/processed.tar -C Data
!cp /content/drive/MyDrive/glioma/manifest.csv Data/
!python -m src.to_npy

## 2. Run: use only what you need this session
Each cell below is independent and assumes Setup was run.

To run a script from the repo, use `!python -m src.<name>` with no `.py`:
- `!python -m src.time_cnn` runs `src/time_cnn.py`
- `!python -m src.train_cnn` runs `src/train_cnn.py` final model mode
- `!python -m src.train_cnn --dev` runs `src/train_cnn.py` tuning mode
- `!python -m src.train_fused` runs `src/train_fused.py`


Run these from the repo root (Setup step 2 puts you there).
Library files such as `src/cnn.py` are imported by the scripts and are not run directly.

In [ ]:
# Confirm a GPU is attached
!nvidia-smi -L

In [ ]:
# Tuning: tries every pair in TUNING_GRID on fold 0 (held-out fold is never predicted)
!python -m src.train_cnn --dev

In [ ]:
# Final run: all folds with the settings in the file (re-run after a disconnect to resume)
!python -m src.train_cnn

In [ ]:
# Training fused model of image features and tabular features (age+sex)
!python -m src.train_fused

## 3. Save results
Results are written to `MyDrive/glioma/cnn`. Download `fold*_pred.csv`, `runs.csv`, and `history.csv` into the repo folder `results/cnn/`
